# ĐỒ ÁN PHÂN TÍCH DỮ LIỆU
## QUY TRÌNH 2.1.2: TIỀN XỬ LÝ DỮ LIỆU - TOÀN VẸN THAM CHIẾU, NGOẠI LAI, DỮ LIỆU THIẾU & TÁI CẤU TRÚC CALENDAR
- **Thành viên thực hiện:** Nguyễn Thị Thanh Hằng
- **Nhiệm vụ theo phân công:**
  1. Xử lý toàn vẹn tham chiếu và tính nhất quán giữa các cột (Khóa liên kết, Group ID, RevExp, BL_ID, dấu Amount).
  2. Xử lý giá trị ngoại lai (IQR trên log10|Amount| theo từng Subgroup).
  3. Xử lý dữ liệu thiếu (Date, Channel, Region, Amount).
  4. Tái cấu trúc bảng Calendar (Dựng lại đầy đủ 365 ngày của năm 2023).
  5. Đánh giá chất lượng sau làm sạch và xuất 10 bảng dữ liệu chuẩn.

In [ ]:
# Kết nối Google Drive nếu đang chạy trên Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass  # Chạy trên môi trường cục bộ (Jupyter Notebook / VS Code)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, spearmanr

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Đường dẫn đến thư mục dữ liệu thô (tự động nhận diện local hoặc Google Colab)
RAW = Path("RAW_DATA/raw") if Path("RAW_DATA/raw").exists() else Path("/content/drive/MyDrive/[LTPTDL] Đồ án/raw")

FILES = [
    "Fact", "Calendar", "Business Line", "Group", "Subgroup",
    "Channel", "Region", "Headlines", "KPI", "Ratios"
]

# Đọc toàn bộ bảng dữ liệu dưới dạng chuỗi
tables = {
    f: pd.read_csv(RAW / f"{f}.csv", dtype=str, encoding="utf-8-sig")
    for f in FILES
}

fact = tables["Fact"]

# Kiểm tra kích thước các bảng
{name: df.shape for name, df in tables.items()}

### Bước tiền đề: Tiếp nhận dữ liệu đã qua xử lý trùng lặp và chuẩn hóa định dạng (Bước 1 & 2)

In [ ]:
# Định nghĩa hàm chuẩn hóa số Amount
def parse_amount(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    neg = s.startswith("(") or s.startswith("-")
    s = re.sub(r"USD|\$|[()\s-]", "", s, flags=re.I)
    if re.fullmatch(r"\d{1,3}(?:\.\d{3})+,\d{2}|\d+,\d{2}", s):   # kiểu VN: 6.720,07
        s = s.replace(".", "").replace(",", ".")
    else:                                                          # kiểu quốc tế: 1,632.79
        s = s.replace(",", "")
    return (-1 if neg else 1) * float(s)

# Tạo bản sao, giữ nguyên dữ liệu gốc
df = fact.copy()

# Đổi tên cột cho dễ thao tác
df = df.rename(columns={
    "Amount, $": "Amount",
    "Business Line ID": "BL_ID",
    "Business Line Name": "BL_Name",
    "Subgroup ID": "Subgroup_ID",
    "Group ID": "Group_ID",
    "Revenue or expense": "RevExp"
})

# Bảng chuẩn dùng để sửa khóa
SG = tables["Subgroup"]
BL = tables["Business Line"].copy()
n_goc = len(df)

# 1. Xóa dòng trùng (Thành viên: Trần Nhựt Hào)
df = df.drop_duplicates().reset_index(drop=True)

# 2. Chuẩn hoá định dạng (Thành viên: Trần Nhựt Hào)
# 2.1 Date:
df["Date"] = pd.to_datetime(
    df["Date"], format="%Y-%m-%d", errors="coerce"
).fillna(
    pd.to_datetime(df["Date"], format="%d/%m/%Y", errors="coerce")
).fillna(
    pd.to_datetime(df["Date"], format="%Y%m%d", errors="coerce")
).fillna(
    pd.to_datetime(df["Date"], format="%b %d, %Y", errors="coerce")
)

# 2.2 Amount: đưa về số thực
df["Amount"] = df["Amount"].map(parse_amount)

# 2.3 Chuẩn hoá nhãn phân loại
df["Channel"] = df["Channel"].str.strip().str.lower().map({
    "online": "Online", "store": "Store", "distributor": "Distributor"
})

df["Region"] = (
    df["Region"].str.strip().str.lower()
    .str.replace("miền ", "", regex=False)
    .map({"bắc": "Bắc", "bac": "Bắc", "trung": "Trung", "nam": "Nam"})
)

df["RevExp"] = (
    df["RevExp"].str.strip().str.lower()
    .map({
        "revenue": "Revenue", "rev": "Revenue",
        "expense": "Expense", "expenses": "Expense", "exp": "Expense"
    })
)

### Bước 3, 4, 5, 6: Xử lý toàn vẹn tham chiếu, Ngoại lai, Dữ liệu thiếu và Tái cấu trúc Calendar

In [ ]:
# 3. Xử lý khoá và thông tin không nhất quán (Thành viên: Nguyễn Thị Thanh Hằng)
# 3.1 Chuẩn bị bảng danh mục
SG = SG.copy()
BL = BL.copy()
SG["Subgroup ID"] = SG["Subgroup ID"].astype(str)
SG["Group ID"] = SG["Group ID"].astype(str)
BL["Business Line ID"] = BL["Business Line ID"].astype(str)
BL["Business Line"] = BL["Business Line"].str.replace("\xa0", " ", regex=False).str.strip()

# 3.2 Loại Subgroup ID không có trong danh mục (không biết là khoản mục nào)
df = df[df["Subgroup_ID"].isin(SG["Subgroup ID"])].copy()

# 3.3 Sửa Group ID theo Subgroup ID (Subgroup.csv là bảng chuẩn)
df["Group_ID"] = df["Subgroup_ID"].map(SG.set_index("Subgroup ID")["Group ID"])

# 3.4 Sửa Revenue/Expense theo Group ID (Group 1 là Revenue, còn lại là Expense)
df["RevExp"] = df["Group_ID"].map({"1": "Revenue", "2": "Expense", "3": "Expense", "4": "Expense"})

# 3.5 Sửa Business Line ID theo tên (tên đúng, ID sai)
df["BL_ID"] = df["BL_Name"].str.strip().map(BL.set_index("Business Line")["Business Line ID"])

# 3.6 Xử lý dấu Amount (Revenue dương, Expense âm)
df["Amount"] = np.where(df["Group_ID"] == "1", df["Amount"].abs(), -df["Amount"].abs())

# 4. Outlier: IQR trên log10|Amount| theo từng Subgroup, đánh dấu thành thiếu (Thành viên: Nguyễn Thị Thanh Hằng)
df["Log_Abs"] = np.log10(df["Amount"].abs().replace(0, np.nan))
q1 = df.groupby("Subgroup_ID")["Log_Abs"].transform(lambda s: s.quantile(0.25))
q3 = df.groupby("Subgroup_ID")["Log_Abs"].transform(lambda s: s.quantile(0.75))
is_out = (df["Log_Abs"] < q1 - 1.5 * (q3 - q1)) | (df["Log_Abs"] > q3 + 1.5 * (q3 - q1))
df.loc[is_out, "Amount"] = np.nan
n_outlier = is_out.sum()                  # đếm số dòng outlier trước khi gộp với Amount thiếu
df = df.drop(columns="Log_Abs")

# 5. Dữ liệu thiếu: (Thành viên: Nguyễn Thị Thanh Hằng)
# 5.1 Date: Xoá
df = df.dropna(subset=["Date"]).copy()
# 5.2 Channel và Region: gán "Unknown"
df["Channel"] = df["Channel"].fillna("Unknown")
df["Region"] = df["Region"].fillna("Unknown")
# 5.3 Amount thiếu và xử lý outlier của Amount: điền trung vị theo Subgroup
df["Amount"] = df["Amount"].fillna(df.groupby("Subgroup_ID")["Amount"].transform("median"))

# 6. Calendar: dựng lại đủ 365 ngày (bảng gốc thiếu 5 ngày), giữ nguyên các cột như bảng gốc (Thành viên: Nguyễn Thị Thanh Hằng)
calendar = pd.DataFrame({"Date": pd.date_range("2023-01-01", "2023-12-31")})
calendar["Year"] = calendar["Date"].dt.year
calendar["Quarter"] = calendar["Date"].dt.quarter
calendar["Month"] = calendar["Date"].dt.month
calendar["Month in text"] = calendar["Date"].dt.month_name()
calendar["Week"] = (calendar["Date"].dt.dayofyear - 1) // 7 + 1
calendar["Day"] = calendar["Date"].dt.day

In [ ]:
# Kiểm tra kết quả Data Cleaning
pd.Series({
    "Số dòng ban đầu": n_goc,
    "Số dòng sau làm sạch": len(df),
    "Số ngày Calendar": len(calendar),
    "Outlier đã xử lý": n_outlier,
    "Ô thiếu còn lại": df.isna().sum().sum(),
    "Dòng trùng còn lại": df.duplicated().sum()
})

,0
Số dòng ban đầu,8726
Số dòng sau làm sạch,8515
Số ngày Calendar,365
Outlier đã xử lý,72
Ô thiếu còn lại,0
Dòng trùng còn lại,0


Nhận xét: Sau bước làm sạch, dữ liệu còn 8.515/8.726 giao dịch, giảm 211 dòng do loại bỏ bản ghi trùng lặp, Subgroup không hợp lệ và Date thiếu. Đồng thời, 72 giá trị ngoại lai đã được xử lý và bảng Calendar được khôi phục đầy đủ 365 ngày. Kết quả kiểm tra cho thấy bảng Fact không còn ô thiếu hoặc dòng trùng lặp, đáp ứng điều kiện để chuyển sang bước tiếp theo.

In [ ]:
# Xuất 10 bảng dữ liệu đã làm sạch (chưa join)
import os

out_dir = str(Path("DATA_PROCESS/data processed") if Path("DATA_PROCESS/data processed").parent.exists() else Path("/content/drive/MyDrive/[LTPTDL] Đồ án/data processed"))
os.makedirs(out_dir, exist_ok=True)

# Lấy 10 bảng gốc, thay 3 bảng đã xử lý
cleaned_tables = dict(tables)
cleaned_tables["Fact"] = df.rename(columns={       # trả tên cột về như bảng gốc
    "Amount": "Amount, $", "BL_ID": "Business Line ID", "BL_Name": "Business Line Name",
    "Subgroup_ID": "Subgroup ID", "Group_ID": "Group ID", "RevExp": "Revenue or expense"
})
cleaned_tables["Calendar"] = calendar
cleaned_tables["Business Line"] = BL

for name, table in cleaned_tables.items():
    table.to_csv(out_dir + f"/{name}.csv", index=False, encoding="utf-8-sig")

print(f"-> Đã xuất đầy đủ 10 bảng dữ liệu đã làm sạch vào thư mục: {out_dir}")

Sau Data Cleaning, nhóm lưu bộ dữ liệu gồm 10 bảng CSV riêng biệt, chưa thực hiện JOIN. Các bảng có vấn đề về chất lượng dữ liệu được xử lý, trong khi những bảng còn lại được giữ nguyên. Bộ dữ liệu này là đầu vào cho giai đoạn Data Integration và các bước phân tích tiếp theo.